In [ ]:
# ============================================================
# English Listening Video Generator (方案 B — Grouped Multi-Character)
# All-in-one cell: Install → Mount Drive → Clone → Run → Download
# ============================================================

# ===== 1. Install Dependencies =====
!apt-get update -qq && apt-get install -y -qq ffmpeg git
!pip install -q kokoro soundfile torch edge-tts Pillow opencc-python-reimplemented
!pip install -q cn2an pypinyin ordered_set jieba
!apt-get install -y -qq fonts-noto-cjk fonts-dejavu-core
import os; os.makedirs('/content/output', exist_ok=True)
print(f'FFmpeg: {os.popen("ffmpeg -version 2>/dev/null | head -1").read().strip()}')
print('Dependencies installed.')

# ===== 2. Mount Google Drive =====
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/listening_videos'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Google Drive: {DRIVE_DIR}')

# ===== 3. Clone Repository from GitHub =====
REPO_URL = 'https://github.com/collinsgraciano/colab_listening_b.git'
REPO_DIR = '/content/listening_b'
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}
required = ['mcp_client.py', 'llm_client.py', 'tts_engine.py', 'timeline.py',
            'grouping_b.py', 'video_compose.py', 'pipeline.py', 'topics.json',
            'topic_manager.py']
missing = [f for f in required if not os.path.exists(os.path.join(REPO_DIR, f))]
if missing:
    print(f'Missing files: {missing}')
else:
    print('All scripts cloned successfully!')

# ===== 4. API Keys =====
# MCP OAuth Tokens — array format, one token per line
# Get tokens from: C:\Users\Administrator\.codely-cli\mcp-oauth-tokens.json
# Add/remove tokens freely; pipeline auto-rotates on 积分 errors
MCP_TOKENS = [
    'PASTE_TOKEN_1_HERE',
    'PASTE_TOKEN_2_HERE',
    # 'PASTE_TOKEN_3_HERE',
]

# SenseNova API Key
SENSENOVA_API_KEY = 'PASTE_YOUR_API_KEY_HERE'

# Join tokens for CLI
MCP_TOKENS_STR = ','.join(t.strip() for t in MCP_TOKENS if t.strip() and not t.strip().startswith('PASTE_'))

if not MCP_TOKENS_STR:
    print('\n⚠️  Please paste your MCP tokens above!')
elif SENSENOVA_API_KEY == 'PASTE_YOUR_API_KEY_HERE':
    print('\n⚠️  Please paste your SenseNova API key above!')
else:
    n_tokens = len([t for t in MCP_TOKENS_STR.split(',') if t.strip()])
    os.environ['SENSENOVA_API_KEY'] = SENSENOVA_API_KEY
    print(f'\n✅ MCP Tokens: {n_tokens} token(s) set')
    print(f'✅ SenseNova API Key set ({len(SENSENOVA_API_KEY)} chars)')

    # ===== 5. Run Pipeline =====
    TOPIC = ''                          # e.g. 'Doing Laundry' or '' for random
    CEFR = 'A2'                         # A1, A2, B1, B2, C1, C2
    NUM_LINES = 18                      # Number of dialogue lines
    STRUCTURE = 'original'             # 'original' or 'enhanced'
    OUTPUT_DIR = DRIVE_DIR             # Save to Google Drive for resume
    RESUME = True                      # Resume from last checkpoint if interrupted

    cmd = f'cd /content/listening_b && python pipeline.py '
    if TOPIC.strip():
        cmd += f'--topic "{TOPIC}" '
    cmd += f'--cefr {CEFR} --num-lines {NUM_LINES} --structure {STRUCTURE} '
    cmd += f'--output "{OUTPUT_DIR}" '
    cmd += f'--mcp-tokens {MCP_TOKENS_STR} --api-key {SENSENOVA_API_KEY}'
    if RESUME:
        cmd += ' --resume'

    print('\nRunning:')
    print(cmd)
    print()
    !{cmd}

    # ===== 6. Archive results to topic-named folder on Google Drive =====
    import json, shutil, re
    script_data = json.load(open(f'{OUTPUT_DIR}/script.json', encoding='utf-8'))
    topic_name = script_data.get('title', 'untitled')
    safe_topic = re.sub(r'[^\w\s-]', '', topic_name).strip().replace(' ', '_')
    ARCHIVE_DIR = f'{DRIVE_DIR}/archive/{safe_topic}'
    os.makedirs(ARCHIVE_DIR, exist_ok=True)

    # Video: rename to YouTube title
    yt_title = script_data.get('youtube_title', '')
    if not yt_title:
        yt_title = topic_name
    # Sanitize title for filename (remove emoji, special chars)
    safe_title = re.sub(r'[\U0001F000-\U0001FFFF]', '', yt_title)  # remove emoji
    safe_title = re.sub(r'[\\/:*?"<>|]', '', safe_title).strip()  # remove invalid filename chars
    vid_src = f'{OUTPUT_DIR}/videos/final_video.mp4'
    if os.path.exists(vid_src):
        shutil.copy2(vid_src, f'{ARCHIVE_DIR}/{safe_title}.mp4')
        print(f'  Copied: {safe_title}.mp4')

    # Other files
    for item in ['script.json', 'thumbnail.jpg', 'youtube_metadata.json',
                 'subtitles/output.srt', 'subtitles/meta.json']:
        src = f'{OUTPUT_DIR}/{item}'
        if os.path.exists(src):
            dst = f'{ARCHIVE_DIR}/{item.replace("subtitles/", "")}'
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            print(f'  Copied: {item}')
    # Copy images
    img_dir = f'{OUTPUT_DIR}/images'
    if os.path.isdir(img_dir):
        for fn in os.listdir(img_dir):
            shutil.copy2(f'{img_dir}/{fn}', f'{ARCHIVE_DIR}/{fn}')
        print(f'  Copied: images/')
    print(f'\n📁 Archived to: {ARCHIVE_DIR}')

    # ===== 7. Download Results =====
    from google.colab import files

    for label, path in [
        ('Video', f'{OUTPUT_DIR}/videos/final_video.mp4'),
        ('Thumbnail', f'{OUTPUT_DIR}/thumbnail.jpg'),
        ('Script', f'{OUTPUT_DIR}/script.json'),
        ('YouTube metadata', f'{OUTPUT_DIR}/youtube_metadata.json'),
    ]:
        if os.path.exists(path):
            size = os.path.getsize(path)
            size_str = f'{size/(1024*1024):.1f}MB' if size > 1024*1024 else f'{size//1024}KB'
            print(f'{label}: {path} ({size_str})')
            files.download(path)
        else:
            print(f'{label}: not found at {path}')

    # ===== 8. Preview Video (optional) =====
    from IPython.display import HTML
    from base64 import b64encode

    video_path = f'{OUTPUT_DIR}/videos/final_video.mp4'
    if os.path.exists(video_path):
        with open(video_path, 'rb') as f:
            video_b64 = b64encode(f.read()).decode()
        html = f'''
        <video width="640" height="360" controls>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
        '''
        display(HTML(html))
    else:
        print('Video not found for preview.')
    print('\n✅ All done!')